In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece fastapi uvicorn pyngrok


In [2]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

# model.eval()
# print("✅ Mistral 7B loaded in 4-bit")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Mistral 7B loaded in 4-bit


In [3]:
# from fastapi import FastAPI
# from pydantic import BaseModel

# app = FastAPI()

# class Payload(BaseModel):
#     transcript: str

# @app.post("/summarize")
# def summarize(data: Payload):

#     prompt = f"""
# You are an expert video summarizer.

# Summarize the following transcript clearly and concisely.
# Focus on key points, structure, and intent.

# Transcript:
# {data.transcript}

# Summary:
# """

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=4096
#     ).to(model.device)

#     with torch.no_grad():
#         output = model.generate(
#             **inputs,
#             max_new_tokens=250,
#             temperature=0.3,
#             do_sample=True,
#             top_p=0.9
#         )

#     summary = tokenizer.decode(
#         output[0],
#         skip_special_tokens=True
#     )

#     # Remove prompt from output
#     summary = summary.split("Summary:")[-1].strip()

#     return {"summary": summary}


In [4]:
# # @title
# from pyngrok import ngrok
# import nest_asyncio
# import uvicorn

# nest_asyncio.apply()

# NGROK_AUTH_TOKEN = "31w02Wf12GP0XnNoCFW1bZDf2AX_2LwtBztP23omxWsDRnfQG"
# ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# public_url = ngrok.connect(8000)
# print("🌍 PUBLIC URL:", public_url)

# uvicorn.run(app, host="0.0.0.0", port=8000)



🌍 PUBLIC URL: NgrokTunnel: "https://db7e-34-125-96-14.ngrok-free.app" -> "http://localhost:8000"


RuntimeError: asyncio.run() cannot be called from a running event loop

In [2]:
pip install -U bitsandbytes

In [5]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True # Add this line
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

app = FastAPI()

class Payload(BaseModel):
    transcript: str

@app.post("/summarize")
def summarize(data: Payload):
    prompt = f"""
Summarize the following transcript clearly and concisely.

Transcript:
{data.transcript}

Summary:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    summary = text.split("Summary:")[-1].strip()
    return {"summary": summary}

Overwriting app.py


In [6]:
from pyngrok import ngrok

ngrok.set_auth_token("31w02Wf12GP0XnNoCFW1bZDf2AX_2LwtBztP23omxWsDRnfQG")
public_url = ngrok.connect(8000)
print("🌍 PUBLIC URL:", public_url)


🌍 PUBLIC URL: NgrokTunnel: "https://1712-34-125-96-14.ngrok-free.app" -> "http://localhost:8000"


In [7]:
!uvicorn app:app --host 0.0.0.0 --port 8000

Loading weights: 100% 291/291 [00:58<00:00,  5.00it/s, Materializing param=model.norm.weight] 
INFO:     Started server process [13969]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
INFO:     2401:4900:84d5:b6f0:218e:4277:962c:cdfb:0 - "POST /summarize HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Finished server process [13969]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "uvloop/loop.pyx", line 1512, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1505, in uvloop.loop.Loop.run_until_complete
  File "